# Train 7.3M ELT Supervised Model (Kaggle T4x2)

This notebook pulls the `elt-supervised-7.3m` repository and begins training using the dual T4 GPUs available on Kaggle. It is pre-configured with optimizations for T4 hardware (e.g. `use_bfloat16: False` to avoid slow emulation).

In [ ]:
!pip install torch torchvision diffusers accelerate tqdm pillow kagglehub

In [ ]:
import os
if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')

!rm -rf elt-supervised-7.3m
!git clone https://github.com/ParthBrijpuria/elt-supervised-7.3m.git
%cd elt-supervised-7.3m

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("greatgamedota/ffhq-face-data-set")

print("Path to dataset files:", path)

## Step 1: Pre-encode all images through the VAE (runs ONCE, ~3 min)

This encodes all 70,000 FFHQ images into VAE latent tensors and saves them to a `.pt` file.
Training then loads directly from RAM — **no VAE, no PNG decoding, no disk I/O** during training.
This gives a ~3-5x speedup per epoch.

**Skip this cell if `latents_ffhq_128.pt` already exists** (e.g., from a previous run).

In [ ]:
import os
if not os.path.exists('latents_ffhq_128.pt'):
    !python preprocess_latents.py --train_dir $path --output latents_ffhq_128.pt --batch_size 128
else:
    print('latents_ffhq_128.pt already exists, skipping preprocessing.')

## Step 2: Train with pre-encoded latents on dual T4 GPUs

In [ ]:
!accelerate launch --num_processes 2 run_train.py --train_dir $path --latent_file latents_ffhq_128.pt --gpu_config gpu_config.json --epochs 1000 --batch_size 128